# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Houssem-Bjaoui/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Framing: a scoring / ranking task, built on top of a binary classification model.**

Clustering does not fit: clustering finds groups when you don't already know what you're looking for, but here the goal is specific — prioritize pages for review. There is nothing to "discover" in the clustering sense.

The real decision is not *"does this one page need a refresh, yes or no?"* — it is *"out of thousands of pages, which ones should a reviewer look at first, given limited time?"* That is a **ranking/scoring** problem, not a plain classification problem, for three reasons:

1. A content team never reviews every page flagged "positive" — they need pages **ordered by priority**, not just split into two buckets.
2. A continuous score (e.g., 0-100) carries more information than a binary label: it can separate "urgent" from "keep an eye on this."
3. The metric that matters for this decision is *"is the top of the list trustworthy?"* (precision at the top, i.e. precision@K) — a question that only makes sense in a ranking setup, not a plain classification one.

In practice, the two framings are not opposites: a **binary classifier** is a natural building block. We can train it on the proxy label described in Section 2, then **sort pages by the model's predicted probability** instead of using the raw yes/no label. Classification supplies the mechanism; ranking/scoring is the actual product a reviewer uses.

**Bottom line:** this lane is a **scoring/ranking task**, implemented with a binary classification model whose output probability becomes the ranking score.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy label used: `trend_direction == "down"`**

This is an **observed outcome**, not a manually defined rule: `trend_direction` is computed from the page's own recent traffic history (comparing a recent window against a prior window), not from someone's opinion of the page. Running the actual starter dataset confirms **16,262 of 30,000 pages (54.2%)** currently carry `trend_direction == "down"` — so a bit over half the pages would be labeled positive under this proxy.

**What this proxy represents:**
A page is flagged when its *observed* performance trend (traffic/impressions pattern over the measured window) is moving downward. It is a real, measurable pattern already present in the data — useful because it does not require anyone to manually label pages as "needs refresh," which would be slow, subjective, and inconsistent across reviewers.

**What this proxy does NOT mean:**

- It does **not** mean the page is broken, low quality, or badly written.
- It does **not** mean refreshing the page will fix the trend — that is a hypothesis to test later, not something the label proves.
- It does **not** mean the decline was *caused* by anything content-related — seasonality, a competitor change, or a SERP feature shift could explain a "down" trend just as easily.
- A `down` label is **not proof** a page must be refreshed. It is only an observable signal that a page **may be** worth a human look, alongside other signals (demand, position, freshness) considered together.

This distinction matters because the proxy is a stand-in for the thing we actually care about ("does this page represent a genuine content opportunity worth review?"), and we do not have a ground-truth label for that — only this observable, imperfect substitute.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Houssem-Bjaoui/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Confirm how the proxy actually appears in the data
print("trend_direction value counts:")
print(df['trend_direction'].value_counts())

print("\ntrend_direction as percentages:")
print((df['trend_direction'].value_counts(normalize=True) * 100).round(1))

# Build the binary proxy target used for modeling
df['is_down'] = (df['trend_direction'] == 'down').astype(int)
print(f"\nProxy base rate (share of pages labeled 'down'): {df['is_down'].mean()*100:.1f}%")


trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction as percentages:
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Proxy base rate (share of pages labeled 'down'): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@K (precision at the top of the ranked list).**

Reasoning from the actual decision being made: a reviewer will only look at the **top N** pages the system surfaces this week — not the whole dataset, and not a simple pass/fail count. What matters is: *of the pages we put at the top of the list, how many actually turn out to be worth reviewing?* That is exactly what precision@K measures. If precision@K is high, a reviewer's limited time is well spent; if it is low, the tool is wasting their time — regardless of how the model performs on pages nobody was ever going to look at anyway.

**Why not plain accuracy:** with **54.2%** of pages already labeled "down" (from Section 2), a model that just guesses the majority class would score around 54% accuracy while being useless for prioritization. Accuracy does not tell us whether the *top* of the ranked list is trustworthy, so it is a poor fit here.

**Cost of the two error types, and why precision is emphasized over recall for this metric choice:**

- **False positive** (page ranked high, but not actually worth reviewing): wastes scarce reviewer time — directly and immediately costly, since review capacity is the bottleneck.
- **False negative** (a genuine opportunity page ranked low or missed): a missed opportunity, but not necessarily gone forever — the page can still surface in a future scoring run, or be caught through other means.

Because reviewer time is the scarcest resource, and false positives burn it directly, **precision at the top of the list matters more than catching every single opportunity** — which is why precision@K is the primary metric rather than recall or plain accuracy.

**Secondary metrics:**

- **Recall@K** — of all the pages that were genuinely worth reviewing, how many did the top-K list actually catch? Tracked so the model isn't optimized into being "safe" by only flagging obvious cases.
- **ROC-AUC or PR-AUC** — a threshold-independent view of how well the model separates the two classes overall, useful for comparing model versions during development, even though it is not the number reviewers will see.

**What "good" would look like:** precision@K meaningfully **above** the proxy base rate (54.2%). If the top of the model's ranked list is only about as reliable as picking pages at random, the model is not adding value over simply reviewing pages in no particular order.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page belonging to one client**, at the time the dataset was captured.

The starter dataset (`data/raw/content_refresh_anonymized.csv`) has **30,000 rows and 44 columns**. `content_id` is confirmed unique across all 30,000 rows (no duplicates), and the pages belong to **32 distinct clients** (`client_id`) — so a single client contributes many rows, one per page they own. This confirms the unit of analysis: each row is one page, described by its own traffic, content, and freshness signals, for one specific client.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Houssem-Bjaoui/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Shape (rows, columns):", df.shape)
print("content_id is unique:", df['content_id'].is_unique)
print("Number of distinct clients:", df['client_id'].nunique())

# Show the unit of analysis clearly with a focused set of real columns
cols_to_show = [
    'content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'clicks_90d',
    'content_age_days', 'trend_direction', 'avg_position', 'ctr', 'word_count',
    'days_since_last_update', 'engagement_rate', 'scroll_rate'
]
df[cols_to_show].head(10)


Shape (rows, columns): (30000, 44)
content_id is unique: True
Number of distinct clients: 32


,content_id,client_id,impressions_90d,sessions_90d,clicks_90d,content_age_days,trend_direction,avg_position,ctr,word_count,days_since_last_update,engagement_rate,scroll_rate
0,content_304f48230142,client_f369cb89fc,3803,17,29,187,down,10.6,0.76,3221.0,20,5.88,4.55
1,content_a1fb4e703a9e,client_4e07408562,15320,9,7,445,down,20.3,0.05,2481.0,25,0.00,10.00
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,141,down,36.5,0.09,3515.0,20,0.00,28.57
3,content_331d6c4de07b,client_19581e27de,11751,78,58,463,stable,6.2,0.49,NaN,22,1.28,3.45
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,24,263,down,44.0,0.13,2803.0,14,0.00,24.29
5,content_d4084a4bc775,client_f369cb89fc,3970,5,1,147,down,8.5,0.03,3080.0,20,0.00,25.00
6,content_9a34b442b552,client_8722616204,20,1,0,90,down,7.0,0.00,3059.0,20,0.00,0.00
7,content_a63219c6e95a,client_19581e27de,1724,28,1,445,stable,21.2,0.06,NaN,22,3.57,7.14
8,content_5e6c160719bc,client_6208ef0f77,32574,68,29,90,down,46.0,0.09,3807.0,20,5.88,6.25
9,content_c27558df2b0c,client_19581e27de,1240,3,2,257,down,4.9,0.16,NaN,104,0.00,0.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A simple fixed rule turns out to be unreliable — and in this data, actually points the wrong way.**

Testing the intuitive rule *"refresh every page older than 365 days"* directly against the actual data: pages older than 365 days decline (`trend_direction == "down"`) **42.6%** of the time, while pages 365 days old or younger decline **57.3%** of the time — compared to an overall base rate of **54.2%**. In other words, in this dataset, *older* pages are **less** likely to be declining than newer ones, the opposite of what the fixed rule assumes. A rule built on age alone would misdirect review effort.

**No single signal dominates either.** Checking the correlation of each observable signal against the decline proxy, every value stays small (roughly between -0.16 and +0.09 — `content_age_days` at -0.16, `word_count` at +0.09, `days_since_last_update` at +0.08, with `ctr`, `avg_position`, `impressions_90d`, `engagement_rate`, and `scroll_rate` all weaker still). No one variable is strong enough to hang a single if-statement threshold on — the signal that predicts decline **may** come from a combination of several weak, partial signals rather than any one obvious cutoff.

**This suggests, carefully:**

- A single manually-chosen threshold (age, position, word count, etc.) is **not** well supported by what is actually observed in this data — the age-365 example shows a naive rule can be directionally wrong, not just imprecise.
- ML **may** offer a more flexible way to combine multiple weak, partial signals into one score than a person manually chaining if-statements could achieve.
- This is a hypothesis, not a guarantee: whether an ML model actually **beats** a well-chosen baseline rule must be **demonstrated through evaluation** (comparing precision@K for the model against precision@K for the best simple rule we can construct), not assumed in advance.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Houssem-Bjaoui/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_down'] = (df['trend_direction'] == 'down').astype(int)

# Test the naive fixed rule: "refresh every page older than 365 days"
older = df[df['content_age_days'] > 365]
younger = df[df['content_age_days'] <= 365]

print(f"Pages older than 365 days: {len(older)} ({len(older)/len(df)*100:.1f}% of all pages)")
print(f"  -> decline rate among them: {older['is_down'].mean()*100:.1f}%")
print(f"Pages 365 days old or younger: {len(younger)} ({len(younger)/len(df)*100:.1f}% of all pages)")
print(f"  -> decline rate among them: {younger['is_down'].mean()*100:.1f}%")
print(f"Overall decline rate (baseline): {df['is_down'].mean()*100:.1f}%")

# Check whether any single numeric signal correlates strongly with the proxy
numeric_cols = ['content_age_days', 'days_since_last_update', 'avg_position',
                 'ctr', 'engagement_rate', 'scroll_rate', 'impressions_90d', 'word_count']
corr_with_proxy = df[numeric_cols + ['is_down']].corr(numeric_only=True)['is_down'].drop('is_down')
print("\nCorrelation of each signal with the decline proxy (is_down):")
print(corr_with_proxy.round(3).sort_values())


Pages older than 365 days: 6360 (21.2% of all pages)
  -> decline rate among them: 42.6%
Pages 365 days old or younger: 23640 (78.8% of all pages)
  -> decline rate among them: 57.3%
Overall decline rate (baseline): 54.2%

Correlation of each signal with the decline proxy (is_down):
content_age_days         -0.164
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
days_since_last_update    0.081
word_count                0.090
Name: is_down, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.